In [1]:
def padding(msg):
    l = len(msg)*8

    padded_msg = msg + b'\x80'

    k = ( 448 - l - 8 ) %512
    padded_msg += b'\x00' * (k // 8)
    padded_msg += l.to_bytes(8, 'big')
    return padded_msg

In [2]:
def parse(padded_msg):
    msg_blocks = [padded_msg[ (n-1)*64 : n*64 ] for n in range(1, len(padded_msg) // 64 +1 ) ]
    return msg_blocks

In [3]:
def ROTR(n, x):
    return ((x >> n) | (x << (32 - n))) & 0xffffffff
def ROTL(n, x):
    return ((x << n) | (x >> (32 - n))) & 0xffffffff
def Ch(x, y, z):
    return (x & y) ^ (~x & z)
def Maj(x, y, z):
    return (x & y) ^ (x & z) ^ (y & z)
def SHR(x, n):
    return x >> n
def SIGMA0(x):
    return ROTR(2, x) ^ ROTR(13, x) ^ ROTR(22, x)
def SIGMA1(x):  
    return ROTR(6, x) ^ ROTR(11, x) ^ ROTR(25, x)
def sigma0(x):
    return ROTR(7, x) ^ ROTR(18, x) ^ SHR(x, 3)
def sigma1(x):
    return ROTR(17, x) ^ ROTR(19, x) ^ SHR(x, 10)

In [4]:
H0 = [
    0x6a09e667,
    0xbb67ae85,
    0x3c6ef372,
    0xa54ff53a,
    0x510e527f,
    0x9b05688c,
    0x1f83d9ab,
    0x5be0cd19
    ]

K = [
0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
]

In [5]:
def sha256(msg):
    
    if not isinstance(msg, bytes):
        msg = msg.encode('utf-8')

    msg = padding(msg)
    msg_blocks = parse(msg)

    N = len(msg_blocks)
    H=H0.copy()

    
    for i in range(N):
        block = msg_blocks[i]
        W=[]
        for t in range(16):
            W += [int.from_bytes(block[t*4:t*4+4], 'big')]
        for t in range(16, 64):
            s0 = sigma0(W[t-15])
            s1 = sigma1(W[t-2])
            W += [ (s1 + W[t-7] + s0 + W[t-16] ) % 0x100000000 ]

        a, b, c, d, e, f, g, h = H
        for t in range(64):
            T1 = ( h + SIGMA1(e) + Ch(e, f, g) + K[t] + W[t] ) % 0x100000000
            T2 = ( SIGMA0(a) + Maj(a, b, c) ) % 0x100000000
            h = g
            g = f
            f = e
            e = (d + T1) % 0x100000000
            d = c
            c = b
            b = a
            a = (T1 + T2) % 0x100000000

        
        for letter in [a, b, c, d, e, f, g, h]:
            H += [ ( H.pop(0) + letter ) % 0x100000000 ]
        
    
    digest = b''.join([h.to_bytes(4, 'big') for h in H])
    return digest.hex()



In [6]:
sha256(b'hello')

'2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824'

In [7]:
import hashlib

hashlib.sha256(b'hello').hexdigest()

'2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824'